# Atribuição golpe → jogador no padel: replicação de Decorte et al. (CVPRW 2024)

**Pergunta.** Dado o instante anotado de um golpe, qual dos 4 jogadores bateu na bola?

**Procedimento.**
1. Bola: TrackNet pré-treinado em tênis, sem fine-tuning, em cada rally.
2. Pose: YOLO11m-pose + ByteTrack nas janelas em torno de cada golpe.
3. Identidade: cada jogador recebe um slot (fundo/frente × esquerda/direita) pela homografia da quadra.
4. Atribuição: voto ponderado pela distância punho–bola. Sem bola na janela, vence a maior velocidade de punho.
5. Avaliação nos 319 golpes com jogador anotado. Referência do artigo: 83,70% (jogador) e 86,83% (dupla).

**Ambiente.** Colab com GPU (*Ambiente de execução → Alterar tipo → T4 GPU*). Os vídeos (16 rallies, 268 MB) e
os pesos são baixados e conferidos por sha256. As anotações convertidas e as homografias da quadra vêm em `sample/`
(origem descrita em `sample/README.md`).

## 0. Ambiente

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/henrique-gerhardt/padel-hit-detection.git"
USAR_DRIVE = False   # True guarda data/cache e outputs no Google Drive (a extração continua se a sessão cair)

ROOT = Path("/content/padel-hit-detection")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))   # pacote padelrep: identidade, atribuição e métricas
print("Projeto em", Path.cwd())

In [ ]:
%pip install -q "ultralytics>=8.3" gdown

In [ ]:
import torch

assert torch.cuda.is_available(), "Sem GPU: troque o ambiente para T4 GPU (Ambiente de execução → Alterar tipo)."
DEVICE = "cuda"      # TrackNet (PyTorch)
POSE_DEVICE = "0"    # Ultralytics: índice da GPU
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
if USAR_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    persist = Path("/content/drive/MyDrive/padel-hit-detection")
    for sub in ("data/cache", "outputs"):
        (persist / sub).mkdir(parents=True, exist_ok=True)
        local = ROOT / sub
        if not local.exists():
            local.parent.mkdir(parents=True, exist_ok=True)
            local.symlink_to(persist / sub)
    print("data/cache e outputs no Drive:", persist)

In [ ]:
import copy
import csv
import hashlib
import json
import time
import urllib.request
from collections import Counter
from dataclasses import replace
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from padelrep.annotations import load_canonical_csv
from padelrep.attribution import AttributionConfig, attribute_hit, window_frames
from padelrep.ball import PrecomputedBallDetector
from padelrep.court import CourtModel
from padelrep.evaluation import evaluate
from padelrep.identity import assign_slots_window
from padelrep.types import SLOTS, FrameObs, HitWindow, team_of
from padelrep.video import append_cache, cached_hit_ids, iter_window, read_cache

## 1. Parâmetros do experimento

Todos os valores do experimento ficam aqui, fixados antes de olhar resultados (sem ajuste fino).

In [ ]:
# Caminhos (relativos à raiz do projeto)
VIDEOS_DIR = Path("data/raw/rallies")
ANNOTATIONS_CSV = Path("sample/annotations/hits.csv")   # 319 golpes rotulados, convertidos para slots
CALIBRATION_DIR = Path("sample/calibration")            # homografia imagem → quadra, uma por rally
BALL_CSV = Path("data/cache/ball_tracks.csv")
WINDOWS_JSONL = Path("data/cache/windows.jsonl")
OUTPUTS = Path("outputs")
WEIGHTS_DIR = Path("third_party/weights")

# Vídeo
FPS = 25                   # todos os vídeos do dataset têm 25 fps
PAD_FRAMES = 12            # frames extraídos de cada lado do golpe

# Pose
POSE_WEIGHTS = WEIGHTS_DIR / "yolo11m-pose.pt"
POSE_TRACKER = "bytetrack.yaml"
POSE_CONF = 0.25
POSE_IMGSZ = 1920          # resolução nativa: em 1280 a dupla do fundo some

# Bola (TrackNet)
TRACKNET_REPO = Path("third_party/TrackNet")
TRACKNET_COMMIT = "730ea17a324321c553c480179caa8ff13e4cf553"
TRACKNET_WEIGHTS = WEIGHTS_DIR / "tracknet_tennis.pt"
TN_W, TN_H = 640, 360      # entrada da rede
TN_BATCH = 16
MAX_JUMP = 100.0           # px em 1280x720: detecção que salta mais que isso das vizinhas é descartada
MAX_GAP = 4                # buracos de até 4 frames são interpolados
MAX_GAP_DIST = 80.0        # px em 1280x720, por frame de buraco
INTERP_CONF = 0.4          # confiança atribuída aos pontos interpolados
BALL_MIN_CONF = 0.3

# Identidade
COURT_MARGIN_M = 0.5       # margem fora das linhas para alguém contar como jogador
SMOOTH_BY_TRACK = True     # slot por voto da trilha do ByteTrack na janela

# Atribuição (experimento principal, E1)
ATTR = AttributionConfig(
    method="wrist_vote",
    window_frames=12,      # 500 ms a 25 fps
    sigma_rel=0.5,         # largura do kernel, em alturas de jogador
    temporal_sigma=None,
    min_kp_conf=0.3,
    min_ball_conf=0.3,
    use_wrist_speed_fallback=True,
    min_wrist_speed=0.05,  # alturas de jogador por frame
)

## 2. Dados

Os vídeos vêm do compartilhamento público indicado no artigo (Nextcloud do imec). Só os 16 rallies com golpes
rotulados são baixados, e cada arquivo é conferido pelo sha256 do vídeo usado na execução de referência.

As anotações e as homografias vêm da pasta `sample/`. O `sample/README.md` descreve como elas foram derivadas do
dataset: o frame do contato, o casamento com os onsets de áudio, o esquema de rótulos e a calibração por câmera.

In [ ]:
DATASET_URL = "https://cloud.ilabt.imec.be/public.php/dav/files/TFimLDWno6W9ED3/rallies/"
RALLIES = {
    "20230528_VIGO_00": "d76f38874086b44575180f12b3c810a7db1f3bb46ede9158c764fa419f83fd19",
    "20230528_VIGO_01": "f60e0f16a2167fef633710f24117e8e57a465e47f760146f161f1f8d4f89b82a",
    "20230528_VIGO_02": "bc1902911732f896a138d4eb061a088b2398e65acf69cde0117a783af8bc4b1f",
    "20230528_VIGO_03": "44c7a1982c2db4a0fc898841f7ba1a0f21d5dc7e950554f3ac7ca6649a6db367",
    "20230528_VIGO_04": "a88d33e579fcbee3f75f39f7ad2be01dd86b282ce8fd44000bf0c8bd716d36c2",
    "20230528_VIGO_05": "61f973110a70042edff9365ce033560eeef395b898b1c4ef75abd0c47e4217dc",
    "20230528_VIGO_06": "c2a1eda8b88658a6dda06059d2548b6b4dcc374335da371c8759ebb99436082e",
    "20230528_VIGO_07": "12d0a3d0972fc34af6636fa5c999f4ca16c1b6b4f0775887b087fadd5ea51dcd",
    "20230528_VIGO_08": "052b1ccbdd7e68360d73d0bfc44274382908cbe8bb207aa4f9c480341fb3c146",
    "20230528_VIGO_09": "77c4fb3dfcbe7dcb61f52265be05d408b680fa5d18c8431ddb24a366057d266c",
    "20230528_VIGO_10": "693eb37b5bf1380cb34641ea4fd0f77c49ca97ebe6425d3eb9160dde69a4f2dc",
    "20230528_VIGO_11": "333509e701996aef16263235013140fc893ecda67040a0c2122d08bb585eed8e",
    "20230528_VIGO_12": "bac2a0a64e66dd4e9e3f2f3a824e2f4f97172bfe11b659f822dd5363b5baa448",
    "20230528_VIGO_14": "af2ebe192371d97a443fd1dfda137720ddf5922ab1ef70fe2c6a7737b32e5324",
    "20230528_VIGO_15": "3197e662ee17508b7ade44e3ce966854a0fe0cf0ece920b3f12dd099deca0439",
    "20230528_VIGO_16": "ec334ba1618562d9cb8144a4febbdeab8a96b5f30eb24b09e4c5df844b39876b",
}


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def fetch(url, dest, expected_sha256):
    # Baixa só se o arquivo não existe ou difere; depois confere o hash.
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not (dest.exists() and sha256(dest) == expected_sha256):
        urllib.request.urlretrieve(url, dest)
    assert sha256(dest) == expected_sha256, f"{dest.name}: sha256 diferente do esperado"
    return dest


def video_path(video_id):
    return VIDEOS_DIR / f"{video_id}.mp4"

In [ ]:
for vid, digest in RALLIES.items():
    fetch(DATASET_URL + vid + ".mp4", video_path(vid), digest)
print(len(RALLIES), "vídeos conferidos")

Golpes rotulados (`sample/annotations/hits.csv`): o slot anotado é a posição do jogador vista da câmera.

In [ ]:
hits = load_canonical_csv(ANNOTATIONS_CSV, only_labeled=True)
assert {h.video_id for h in hits} == set(RALLIES)
print(len(hits), "golpes rotulados em", len(RALLIES), "rallies")
print("por slot:", dict(sorted(Counter(h.gt_slot for h in hits).items())))
print("por rally:", dict(sorted(Counter(h.video_id[-7:] for h in hits).items())))

Homografias (`sample/calibration/`): o contorno da quadra projetado na imagem confere a calibração.

In [ ]:
courts = {vid: CourtModel.load(CALIBRATION_DIR / f"{vid}.json") for vid in RALLIES}

example = hits[0]
_, frame = next(iter_window(video_path(example.video_id), example.frame, 0))
cv2.polylines(frame, [courts[example.video_id].court_polygon_image().astype(np.int32)], True, (0, 255, 255), 3)
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f"{example.hit_id}: quadra pela homografia")
plt.axis("off")
plt.show()

## 3. Pesos dos modelos

- `yolo11m-pose.pt`: release oficial da Ultralytics.
- TrackNet: reimplementação PyTorch de [yastrebksv/TrackNet](https://github.com/yastrebksv/TrackNet), com os pesos
  de tênis indicados no README do repositório, sem fine-tuning.

In [ ]:
fetch("https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11m-pose.pt", POSE_WEIGHTS,
      "29b17eaf3a3117cbea906090dbedf9159f7c6a49db58ec8b99ed2dfde1cf6eb2")

if not TRACKNET_REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/yastrebksv/TrackNet.git", str(TRACKNET_REPO)], check=True)
subprocess.run(["git", "-C", str(TRACKNET_REPO), "checkout", "-q", TRACKNET_COMMIT], check=True)
if not TRACKNET_WEIGHTS.exists():
    subprocess.run(["gdown", "1XEYZ4myUN7QT-NeBYJI0xteLsvs-ZAOl", "-O", str(TRACKNET_WEIGHTS)], check=True)
assert sha256(TRACKNET_WEIGHTS) == "c735bc1a1b13a35f179c6492f778ef4ebb9bffd512a96f4d970b32e076653076"
print("pesos conferidos")

## 4. Bola (TrackNet)

O TrackNet recebe 3 frames consecutivos em 640×360 e devolve um mapa de calor. O pós-processamento segue o
repositório [yastrebksv/TrackNet](https://github.com/yastrebksv/TrackNet):

1. A bola é o pico do mapa (limiar 127 + HoughCircles). A confiança é o valor do mapa no pico.
2. Detecções que saltam mais de `MAX_JUMP` das vizinhas são descartadas.
3. Buracos de até `MAX_GAP` frames são interpolados.

Cada rally é processado inteiro, para o pós-processamento ter contexto. Os frames são lidos em fluxo, e só um lote
fica na memória.

In [ ]:
def read_small(path):
    # Frames do vídeo já redimensionados para a entrada da rede, em float32 [0, 1].
    cap = cv2.VideoCapture(str(path))
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                return
            yield cv2.resize(frame, (TN_W, TN_H)).astype(np.float32) / 255.0
    finally:
        cap.release()


def triplets(frames, batch):
    # Lotes de entradas (t, t-1, t-2) empilhadas nos canais.
    last, buf = [], []
    for frame in frames:
        last = (last + [frame])[-3:]
        if len(last) == 3:
            buf.append(np.concatenate((last[2], last[1], last[0]), axis=2))
            if len(buf) == batch:
                yield np.stack(buf)
                buf = []
    if buf:
        yield np.stack(buf)


def heatmap_peak(heat):
    # Mapa uint8 (360x640) -> (x, y, conf) na escala 640x360, ou None.
    _, binary = cv2.threshold(heat, 127, 255, cv2.THRESH_BINARY)
    circles = cv2.HoughCircles(binary, cv2.HOUGH_GRADIENT, dp=1, minDist=1, param1=50, param2=2,
                               minRadius=2, maxRadius=7)
    if circles is None or len(circles) != 1:
        return None
    x, y = float(circles[0][0][0]), float(circles[0][0][1])
    xi, yi = int(np.clip(round(x), 0, TN_W - 1)), int(np.clip(round(y), 0, TN_H - 1))
    conf = float(heat[max(0, yi - 2):yi + 3, max(0, xi - 2):xi + 3].max()) / 255.0
    return x, y, conf


def track_video(path, model, max_frames=None):
    # Trajetória bruta: uma entrada por frame, (x, y, conf) na resolução original ou None.
    cap = cv2.VideoCapture(str(path))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    frames = read_small(path)
    if max_frames:
        frames = (f for _, f in zip(range(max_frames), frames))
    track = [None, None]   # os dois primeiros frames não têm 3 frames de histórico
    with torch.no_grad():
        for batch in triplets(frames, TN_BATCH):
            inp = torch.from_numpy(np.ascontiguousarray(batch.transpose(0, 3, 1, 2))).to(DEVICE)
            out = model(inp).argmax(dim=1).cpu().numpy().astype(np.uint8)
            for heat in out:
                peak = heatmap_peak(heat.reshape(TN_H, TN_W))
                track.append(None if peak is None else (peak[0] * w / TN_W, peak[1] * h / TN_H, peak[2]))
    return track, (w, h)


def remove_jumps(track, max_jump):
    # Apaga detecções que saltam mais que max_jump em relação a todas as vizinhas detectadas (até 3 frames).
    out = list(track)
    for i, p in enumerate(track):
        if p is None:
            continue
        prev = next((track[j] for j in range(i - 1, max(-1, i - 4), -1) if track[j] is not None), None)
        nxt = next((track[j] for j in range(i + 1, min(len(track), i + 4)) if track[j] is not None), None)
        far = [q is not None and np.hypot(p[0] - q[0], p[1] - q[1]) > max_jump for q in (prev, nxt)]
        known = [q is not None for q in (prev, nxt)]
        if any(known) and all(f for f, k in zip(far, known) if k):
            out[i] = None
    return out


def interpolate_gaps(track, max_gap, max_dist):
    # Preenche buracos curtos por interpolação linear entre as detecções das pontas.
    out = list(track)
    idx = [i for i, p in enumerate(track) if p is not None]
    for a, b in zip(idx, idx[1:]):
        gap = b - a - 1
        if 0 < gap <= max_gap:
            pa, pb = track[a], track[b]
            if np.hypot(pa[0] - pb[0], pa[1] - pb[1]) <= max_dist * (gap + 1):
                for k in range(1, gap + 1):
                    t = k / (gap + 1)
                    out[a + k] = (pa[0] + t * (pb[0] - pa[0]), pa[1] + t * (pb[1] - pa[1]), INTERP_CONF)
    return out


def clean_track(raw, width):
    scale = width / 1280.0   # limiares definidos em 1280x720
    return interpolate_gaps(remove_jumps(raw, MAX_JUMP * scale), MAX_GAP, MAX_GAP_DIST * scale)

In [ ]:
sys.path.insert(0, str(TRACKNET_REPO))
from model import BallTrackerNet   # do repositório TrackNet

tracknet = BallTrackerNet()
tracknet.load_state_dict(torch.load(TRACKNET_WEIGHTS, map_location="cpu", weights_only=True))
tracknet = tracknet.to(DEVICE).eval()

Teste num trecho de um rally antes de rodar tudo: detecções sobre o último frame do trecho.

In [ ]:
vid = hits[0].video_id
raw, (w, h) = track_video(video_path(vid), tracknet, max_frames=150)
clean = clean_track(raw, w)
print(f"{vid}: {len(raw)} frames | bola detectada em {np.mean([p is not None for p in raw]):.0%} "
      f"| após limpeza {np.mean([p is not None for p in clean]):.0%}")

_, frame = next(iter_window(video_path(vid), len(raw) - 1, 0))
pts = np.array([p[:2] for p in clean if p is not None])
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.scatter(pts[:, 0], pts[:, 1], c=np.arange(len(pts)), cmap="autumn", s=12)
plt.title(f"{vid}: trajetória da bola nos primeiros {len(raw)} frames")
plt.axis("off")
plt.show()

Todos os rallies. O progresso vai para `ball_tracks.csv.done`. Se a execução parar, a célula retoma do próximo
rally.

In [ ]:
BALL_CSV.parent.mkdir(parents=True, exist_ok=True)
done_file = BALL_CSV.with_suffix(".csv.done")
done = set(done_file.read_text().split()) if done_file.exists() and BALL_CSV.exists() else set()
if done:   # descarta linhas de um rally interrompido no meio
    with open(BALL_CSV, newline="") as fh:
        rows = [r for r in csv.reader(fh)]
    with open(BALL_CSV, "w", newline="") as fh:
        csv.writer(fh).writerows([rows[0]] + [r for r in rows[1:] if r and r[0] in done])
else:
    with open(BALL_CSV, "w", newline="") as fh:
        csv.writer(fh).writerow(["video_id", "frame", "x", "y", "conf"])
    done_file.write_text("")

for vid in RALLIES:
    if vid in done:
        continue
    t0 = time.time()
    raw, (w, h) = track_video(video_path(vid), tracknet)
    clean = clean_track(raw, w)
    with open(BALL_CSV, "a", newline="") as fh:
        csv.writer(fh).writerows([vid, f, round(p[0], 1), round(p[1], 1), round(p[2], 3)]
                                 for f, p in enumerate(clean) if p is not None)
    with open(done_file, "a") as fh:
        fh.write(vid + "\n")
    print(f"{vid}: {len(raw)} frames | detectado {np.mean([p is not None for p in raw]):.0%} "
          f"| após limpeza {np.mean([p is not None for p in clean]):.0%} | {time.time() - t0:.0f}s")
print("->", BALL_CSV)

## 5. Pose (YOLO11m-pose + ByteTrack)

Para cada golpe, o YOLO roda nos `2 × PAD_FRAMES + 1` frames centrados no instante anotado. O rastreador é
reiniciado a cada golpe, e cada pessoa sai com bbox, 17 keypoints e ID da trilha. As janelas vão para um cache
JSONL, uma linha por golpe, e golpes já presentes no cache são pulados.

In [ ]:
COLORS = {"far_left": (255, 128, 0), "far_right": (0, 200, 255), "near_left": (0, 255, 0), "near_right": (255, 0, 255)}


def extract_window(hit, pose):
    pose.reset()
    frames = [FrameObs(idx, pose(img), None) for idx, img in iter_window(video_path(hit.video_id), hit.frame, PAD_FRAMES)]
    return HitWindow(hit, frames)


def draw_hit(window, court, pred=None):
    # Frame do golpe: quadra, bola na janela, pessoas (cor do slot; cinza = sem slot), punhos, anotação e previsão.
    _, img = next(iter_window(video_path(window.hit.video_id), window.hit.frame, 0))
    cv2.polylines(img, [court.court_polygon_image().astype(np.int32)], True, (0, 255, 255), 2)
    for f in window.frames:
        if f.ball:
            at_hit = f.frame_idx == window.hit.frame
            cv2.circle(img, tuple(map(int, f.ball.xy)), 6 if at_hit else 3, (0, 0, 255) if at_hit else (0, 0, 160), -1)
    by_idx = {f.frame_idx: f for f in window.frames}
    for pl in by_idx.get(window.hit.frame, window.frames[len(window.frames) // 2]).players:
        col = COLORS.get(pl.slot, (128, 128, 128))
        x1, y1, x2, y2 = map(int, pl.bbox)
        cv2.rectangle(img, (x1, y1), (x2, y2), col, 2)
        cv2.putText(img, f"{pl.slot or '-'} #{pl.track_id}", (x1, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
        for wpt in pl.wrists():
            cv2.circle(img, tuple(map(int, wpt)), 5, col, -1)
    txt = f"Anotado: {window.hit.gt_slot}"
    if pred is not None:
        txt += f" | Previsto: {pred.pred_slot or '-'} ({pred.method_used})"
    cv2.putText(img, txt, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def show(images, titles, ncols=1, width=12):
    nrows = -(-len(images) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(width, 0.6 * width * nrows / ncols))
    for ax, img, title in zip(np.atleast_1d(axes).ravel(), images, titles):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
from padelrep.pose import UltralyticsPoseTracker

pose = UltralyticsPoseTracker(str(POSE_WEIGHTS), POSE_TRACKER, POSE_CONF, POSE_IMGSZ, POSE_DEVICE)

Teste em um golpe: pessoas detectadas por frame e o frame do golpe (ainda sem slots, por isso em cinza).

In [ ]:
t0 = time.time()
test_window = extract_window(hits[0], pose)
print(f"{hits[0].hit_id}: {len(test_window.frames)} frames em {time.time() - t0:.1f}s | pessoas por frame:",
      [len(f.players) for f in test_window.frames])
show([draw_hit(test_window, courts[hits[0].video_id])], [hits[0].hit_id])

Todos os golpes.

In [ ]:
WINDOWS_JSONL.parent.mkdir(parents=True, exist_ok=True)
done = cached_hit_ids(WINDOWS_JSONL)
todo = [h for h in hits if h.hit_id not in done]
print(f"{len(hits)} golpes; {len(done)} já no cache; processando {len(todo)}")
t0 = time.time()
for n, hit in enumerate(sorted(todo, key=lambda h: (h.video_id, h.frame)), 1):
    append_cache(WINDOWS_JSONL, extract_window(hit, pose))
    if n % 25 == 0 or n == len(todo):
        print(f"{n}/{len(todo)} ({time.time() - t0:.0f}s)")

## 6. Identidade: slots pela homografia

Cada janela recebe a bola do CSV, com confiança mínima de `BALL_MIN_CONF`. Depois, cada pessoa recebe um slot pela
posição dos pés na quadra: lado da rede e metade esquerda/direita. Pessoas fora da quadra (árbitro, público) são
descartadas, e o slot de cada trilha do ByteTrack é decidido por voto na janela.

In [ ]:
def attach_ball(windows, ball_csv):
    # ball_csv=None remove a bola (experimento E4).
    det = PrecomputedBallDetector(ball_csv, BALL_MIN_CONF) if ball_csv else None
    for w in windows:
        for f in w.frames:
            f.ball = det(None, f.frame_idx, w.hit.video_id) if det else None
    return windows


def assign_slots(windows):
    return [assign_slots_window(w, courts[w.hit.video_id], COURT_MARGIN_M, SMOOTH_BY_TRACK) for w in windows]


def coverage(raw_windows, slotted, n_frames):
    # Medidas da extração nas mesmas janelas usadas na atribuição.
    raw = [len(f.players) for w in raw_windows for f in window_frames(w.frames, w.hit.frame, n_frames)]
    ball, filled, gt_present, any_ball = [], [], [], []
    for w in slotted:
        frames = window_frames(w.frames, w.hit.frame, n_frames)
        any_ball.append(any(f.ball is not None for f in frames))
        for f in frames:
            slots = {p.slot for p in f.players if p.slot}
            ball.append(f.ball is not None)
            filled.append(len(slots))
            gt_present.append(w.hit.gt_slot in slots)
    return {
        "frames com bola": np.mean(ball),
        "golpes com bola em algum frame": np.mean(any_ball),
        "pessoas detectadas por frame": np.mean(raw),
        "jogadores na quadra por frame": np.mean(filled),
        "frames com os 4 slots": np.mean(np.array(filled) == 4),
        "frames com o golpeador presente": np.mean(gt_present),
    }

In [ ]:
raw_windows = attach_ball(read_cache(WINDOWS_JSONL), BALL_CSV)
assert len(raw_windows) == len(hits)
windows = assign_slots(copy.deepcopy(raw_windows))

cov = coverage(raw_windows, windows, ATTR.window_frames)
for name, value in cov.items():
    print(f"{name:34} {value:.2f}" if "por frame" in name else f"{name:34} {value:.1%}")

In [ ]:
w0 = next(w for w in windows if w.hit.hit_id == hits[0].hit_id)
show([draw_hit(w0, courts[w0.hit.video_id])], [f"{w0.hit.hit_id}: slots atribuídos e bola na janela"])

## 7. Atribuição: experimento principal (E1)

Para cada frame da janela de `ATTR.window_frames` frames com bola, cada jogador recebe o voto
`exp(-d² / 2σ²)`. Nele, `d` é a menor distância punho–bola dividida pela altura da bbox do jogador, e `σ` é
`ATTR.sigma_rel`. Vence o jogador com maior soma de votos. Sem bola na janela, vence a maior velocidade de punho.
Um golpe sem previsão conta como erro. Os intervalos de confiança (95%) vêm de bootstrap.

In [ ]:
def run_variant(windows, cfg, tag):
    # Atribui todos os golpes, salva previsões e métricas em outputs/ e devolve (métricas, resultados).
    results = [attribute_hit(w, cfg) for w in windows]
    metrics = evaluate(results, [w.hit for w in windows])
    OUTPUTS.mkdir(exist_ok=True)
    with open(OUTPUTS / f"predictions_{tag}.csv", "w", newline="") as fh:
        wr = csv.writer(fh)
        wr.writerow(["hit_id", "video_id", "frame", "gt_slot", "pred_slot", "method_used", "n_ball_frames"])
        for w, r in zip(windows, results):
            wr.writerow([w.hit.hit_id, w.hit.video_id, w.hit.frame, w.hit.gt_slot or "",
                         r.pred_slot or "", r.method_used, r.n_ball_frames])
    (OUTPUTS / f"metrics_{tag}.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
    return metrics, results


def pct(x):
    return f"{100 * x:.1f}%"

In [ ]:
main, results = run_variant(windows, ATTR, "main")
paper = main["paper"]
print(f"Golpes avaliados: {main['n_hits']} (artigo: {paper['n_hits']}) | cobertura {pct(main['coverage'])}")
for key, name in (("player_acc", "jogador"), ("team_acc", "dupla")):
    lo, hi = main[key + "_ci95"]
    print(f"Acurácia {name:8} {pct(main[key])}  IC95 [{pct(lo)}; {pct(hi)}]  | artigo {pct(paper[key])}  "
          f"| diferença {100 * (main[key] - paper[key]):+.1f} p.p.")
print("\nPor método usado:")
for method, m in main["by_method"].items():
    print(f"  {method:12} n={m['n']:3}  jogador {pct(m['player_acc'])}  dupla {pct(m['team_acc'])}")

## 8. Ablação (E2–E4)

Cada variante muda uma única coisa em relação ao E1, sem ajuste fino.

| Grupo | Variantes |
|---|---|
| E2, método | centro da bbox em vez do punho; peso linear da eq. 1 do artigo |
| E3, σ | 0,25 e 1,0 |
| E3, janela | 6 e 24 frames |
| Fallback | sem o fallback de velocidade do punho |
| E4 | sem bola (só a velocidade do punho) |

In [ ]:
VARIANTS = {
    "main": ATTR,
    "bbox_center": replace(ATTR, method="bbox_center"),
    "paper_eq1": replace(ATTR, method="paper_eq1"),
    "no_fallback": replace(ATTR, use_wrist_speed_fallback=False),
    "sigma_0.25": replace(ATTR, sigma_rel=0.25),
    "sigma_1.0": replace(ATTR, sigma_rel=1.0),
    "win_6": replace(ATTR, window_frames=6),
    "win_24": replace(ATTR, window_frames=24),
}

In [ ]:
ablation = {tag: run_variant(copy.deepcopy(windows), cfg, tag)[0] for tag, cfg in VARIANTS.items()}
ablation["no_ball"] = run_variant(attach_ball(copy.deepcopy(windows), None), ATTR, "no_ball")[0]

print(f"{'Variante':12} {'Jogador':>8} {'Dupla':>8} {'Cobertura':>10}")
for tag, m in ablation.items():
    print(f"{tag:12} {pct(m['player_acc']):>8} {pct(m['team_acc']):>8} {pct(m['coverage']):>10}")

## 9. Análise de erros

Os erros do E1 são classificados pela primeira regra que casar:

1. O golpeador anotado aparece em menos da metade dos frames da janela.
2. A previsão veio do fallback sem bola.
3. A dupla está certa, mas o lado está errado (troca dentro da dupla).
4. A dupla está errada, mas o lado é o mesmo: a mesma coluna, típico de bola alta que, na imagem, cai sobre o
   jogador do fundo.
5. A dupla e o lado estão errados (diagonal).

In [ ]:
def error_category(window, result, n_frames):
    gt, pred = window.hit.gt_slot, result.pred_slot
    frames = window_frames(window.frames, window.hit.frame, n_frames)
    presence = sum(any(p.slot == gt for p in f.players) for f in frames) / max(1, len(frames))
    if not pred:
        return "sem_previsao"
    if presence < 0.5:
        return "golpeador_nao_detectado"
    if result.method_used == "wrist_speed":
        return "fallback_sem_bola"
    if team_of(pred) == team_of(gt):
        return "troca_esq_dir_na_dupla"
    if pred.split("_")[1] == gt.split("_")[1]:
        return "outra_dupla_mesma_coluna"
    return "outra_dupla_diagonal"

In [ ]:
errors = [(w, r) for w, r in zip(windows, results) if r.pred_slot != w.hit.gt_slot]
categories = Counter(error_category(w, r, ATTR.window_frames) for w, r in errors)
print(f"{len(errors)} erros em {len(windows)} golpes")
for cat, n in categories.most_common():
    print(f"  {cat:26} {n}")

labels = list(SLOTS) + ["none"]
conf = np.array([[main["confusion"][gt][p] for p in labels] for gt in SLOTS])
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(conf, cmap="Blues")
for i in range(conf.shape[0]):
    for j in range(conf.shape[1]):
        ax.text(j, i, conf[i, j], ha="center", va="center", color="white" if conf[i, j] > conf.max() / 2 else "black")
ax.set_xticks(range(len(labels)), labels, rotation=30)
ax.set_yticks(range(len(SLOTS)), SLOTS)
ax.set_xlabel("previsto")
ax.set_ylabel("anotado")
ax.set_title("Matriz de confusão (E1)")
plt.tight_layout()
plt.show()

Exemplos: três acertos e três erros, os mesmos golpes das galerias dos slides. Os slots estão coloridos, os
punhos aparecem como pontos e a bola na janela em vermelho.

In [ ]:
by_id = {w.hit.hit_id: (w, r) for w, r in zip(windows, results)}
HITS_OK = ["20230528_VIGO_01_h003", "20230528_VIGO_06_h014", "20230528_VIGO_15_h022"]
HITS_ERR = ["20230528_VIGO_06_h023", "20230528_VIGO_04_h007", "20230528_VIGO_15_h024"]
for name, hit_ids in (("acerto", HITS_OK), ("erro", HITS_ERR)):
    images = []
    for hit_id in hit_ids:
        w, r = by_id[hit_id]
        images.append(draw_hit(w, courts[w.hit.video_id], r))
    show(images, [f"{name}: {h}" for h in hit_ids], ncols=3, width=24)

## 10. Comparação com a execução de referência

A execução de referência é a que gerou a tabela do README (Apple M4, GPU MPS). A pose e a bola foram
recalculadas aqui em outra GPU, então diferenças de alguns décimos são esperadas.

In [ ]:
REFERENCE = {   # acurácia (%) jogador, dupla na execução de referência
    "main": (82.4, 87.8),
    "bbox_center": (73.0, 81.2),
    "paper_eq1": (74.3, 80.6),
    "no_fallback": (78.4, 83.1),
    "sigma_0.25": (82.8, 87.8),
    "sigma_1.0": (81.5, 87.5),
    "win_6": (83.7, 90.0),
    "win_24": (75.2, 80.3),
    "no_ball": (77.1, 85.0),
}

print(f"{'Variante':12} {'Jogador':>8} {'Ref.':>6} {'Dupla':>8} {'Ref.':>6}")
for tag, (ref_player, ref_team) in REFERENCE.items():
    m = ablation[tag]
    print(f"{tag:12} {100 * m['player_acc']:8.1f} {ref_player:6.1f} {100 * m['team_acc']:8.1f} {ref_team:6.1f}")

In [ ]:
subprocess.run(["zip", "-qr", "/content/outputs_colab.zip", str(OUTPUTS), str(BALL_CSV)], check=True)
from google.colab import files

files.download("/content/outputs_colab.zip")